# 第2回 宿題（発展）: 用語集の Q&A で LoRA 学習し、学習前・学習後・用語集検索を比べる

演習のノートブック `notebooks/lora_format_tuning.ipynb` をもとに、題材を架空 FAQ から講座の用語集に差し替えたものです。モデル、LoRA の設定、生成設定は演習と同じで、データと評価だけを変えています。

**このノートブックで確かめること**

1. 用語集の定義文を学習すると、答えの形や講座固有の語への答えがどう変わるか（セクション 5）。
2. 用語集にない語を聞かれたとき、学習後のモデルは「記載がない」と答えられるか、それとも言い切ってしまうか（セクション 5）。
3. 学習せずに用語集を検索して渡す方法と比べて、学習のメリットとデメリットは何か（セクション 3、5）。

**評価の 3 つの区分**（各 7 問、計 21 問）

| 区分 | 内容 | 正解の条件 |
| --- | --- | --- |
| `in` | 用語集にある語を、学習データとは別の聞き方で聞く | 定義の要点の語句を含む |
| `paraphrase` | 別名や説明文から聞く | 定義の要点の語句を含む |
| `out` | 用語集にない語（実在の語と架空の語）を聞く | 「記載がありません」などと答える |

**使うもの**: Google Colab（T4 GPU、無料枠）。モデルは `Qwen/Qwen3.5-0.8B`。データは次の 3 ファイルで、セクション 2 でアップロードします。

- `glossary_sft.jsonl`: `scripts/make_sft_dataset.py` で用語集から作った学習データ（116 件）
- `glossary_eval.json`: 評価用の質問と正解（21 問）
- `glossary.json`: 用語集そのもの（`data/glossary.json`）。用語集検索の比較に使います

セルは上から順に実行してください。

## 0. 環境の確認とライブラリの導入

**やること**: GPU が割り当てられているかを確認し、ライブラリを動作確認済みの版に固定して導入します。

**見るところ**

- 1 つ目のセルに `Tesla T4` と表示されること。
- 2 つ目のセルの最後に、版の一覧（`transformers: 5.18.0` など）が表示されること。導入は 1 分ほどかかります。

**うまくいかないとき**

- 「GPU が見つかりません」と出る: メニューの「ランタイム」→「ランタイムのタイプを変更」→「T4 GPU」を選び、セルを実行し直します。混雑で割り当てられないときは、時間をおいて試します。
- pip の赤い `ERROR: pip's dependency resolver ...`（`cudf` や `pyarrow` に関するもの）は、Colab に最初から入っている別のパッケージとの食い違いの警告です。この演習には影響しないので、そのまま進めます。

In [ ]:
import subprocess
try:
    gpu_info = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout
except FileNotFoundError:   # CPU ランタイムには nvidia-smi が無い
    gpu_info = ""
print(gpu_info or "GPU が見つかりません。ランタイム → ランタイムのタイプを変更 → T4 GPU を選んでください。")

In [ ]:
# Colab の T4（Python 3.13、torch 2.11.0+cu130）で動作確認した版。torch は Colab 同梱のものをそのまま使う
PINS = {
    "transformers": "5.18.0",   # Qwen3.5 は transformers 5 系が必要
    "peft": "0.21.1",           # LoRA の実装
    "trl": "1.14.2",            # 学習ループ（SFTTrainer）
    "accelerate": "1.15.0",
    "datasets": "5.1.0",
}
spec = " ".join(f'"{k}=={v}"' for k, v in PINS.items())
%pip uninstall -y -q torchao   # Colab 同梱の torchao（0.10）は peft 0.21 と非互換で、学習時に ImportError になる。使わないので外す
%pip install -q {spec}

# cudf-cu13 と pyarrow の依存警告が ERROR 表示されるが、cudf は使わないので無視してよい
import transformers, peft, trl, accelerate, datasets, torch
print({m.__name__: m.__version__ for m in [transformers, peft, trl, accelerate, datasets, torch]})

## 1. 設定

演習の `CONFIG` から 2 か所だけ変えています。

- `num_train_epochs: 5`: 学習データが 116 件あり、1 エポックが 15 ステップになります。演習の 20 エポックだと約 290 ステップになり時間がかかりすぎるので、5 エポック（75 ステップ）に減らしています。
- `max_new_tokens: 200`: 用語集の定義文は FAQ の回答より長い（平均 105 文字）ので、96 から増やしています。

**見るところ**: 設定の JSON が表示されます。

In [ ]:
from pathlib import Path
import os, json, time, re
from transformers import set_seed

CONFIG = {
    "base_model": "Qwen/Qwen3.5-0.8B",
    "revision": "2fc06364715b967f1860aea9cf38778875588b17",
    "seed": 42,
    "num_train_epochs": 5,   # 116 件 / 1 ステップ 8 件 = 1 エポック 15 ステップ。5 エポックで 75 ステップ
    "warmup_steps": 6,
    "lr": 2e-4,
    "lora_r": 16,
    "lora_alpha": 32,
    "max_seq_len": 512,
    "max_new_tokens": 200,   # 定義文は FAQ の回答より長いので 96 から増やす
    "gen": {"do_sample": False},
    "work_dir": "/content/glossary_lab",
    "drive_dir": "/content/drive/MyDrive/aie2026/glossary_lab",
}
set_seed(CONFIG["seed"])
WORK = Path(CONFIG["work_dir"]); WORK.mkdir(parents=True, exist_ok=True)

USE_DRIVE = False  # True にすると Google Drive をマウントして adapter と出力を保存する
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    Path(CONFIG["drive_dir"]).mkdir(parents=True, exist_ok=True)
print(json.dumps(CONFIG, ensure_ascii=False, indent=2))

## 2. データの読み込み

**やること**: 学習データ、評価データ、用語集の 3 ファイルを `/content` に置きます。左のファイル一覧にドラッグしてもよく、無ければ下のセルでアップロードを求められます。

- 学習データの各行は演習の `to_chat` と同じ `messages` 形式なので、そのまま `Dataset.from_list` に渡せます。学習データには system メッセージがありません。評価も同じ形にそろえ、user の質問だけを渡します。
- 評価の質問が学習データと同じ文になっていると、覚えた文を答えるだけになります。下のセルで重複が無いことを確かめます。

**見るところ**: `train=116 test=21` と、区分ごとの問数が表示されます。

In [ ]:
DATA_DIR = Path("/content")
NEEDED = ["glossary_sft.jsonl", "glossary_eval.json", "glossary.json"]
missing = [n for n in NEEDED if not (DATA_DIR / n).exists()]
if missing:
    from google.colab import files
    print("次のファイルをアップロードしてください:", missing)
    for name, data in files.upload().items():
        (DATA_DIR / name).write_bytes(data)
missing = [n for n in NEEDED if not (DATA_DIR / n).exists()]
assert not missing, f"ファイルが見つかりません: {missing}"

train_rows = [json.loads(l) for l in (DATA_DIR / "glossary_sft.jsonl").read_text(encoding="utf-8-sig").splitlines() if l.strip()]
EVAL = json.loads((DATA_DIR / "glossary_eval.json").read_text(encoding="utf-8"))
TESTS = EVAL["questions"]
REFUSAL = EVAL["refusal_markers"]   # これらの語を含む答えを「記載がない」と答えたとみなす
TERMS = json.loads((DATA_DIR / "glossary.json").read_text(encoding="utf-8"))["terms"]

train_qs = {r["messages"][0]["content"] for r in train_rows}
leak = [t["q"] for t in TESTS if t["q"] in train_qs]
assert not leak, f"評価の質問が学習データと同じ文です: {leak}"
print(f"train={len(train_rows)} test={len(TESTS)}", {ty: sum(t["type"] == ty for t in TESTS) for ty in ["in", "paraphrase", "out"]})
print("学習例の 1 件:", train_rows[0]["messages"])

## 3. モデルの読み込みと学習前の評価、用語集検索の評価

**やること**: 演習と同じく量子化なしでモデルを読み込み、21 問に答えさせます。同じ学習前のモデルで、次の 2 つの条件を測ります。

- `before`: 質問だけを渡す。学習前の基準です。
- `search`: 質問の中に用語集の見出し語や別名が含まれていれば、その項目の定義を最大 3 件プロンプトに入れて渡す。学習せずに知識を与える方法の比較対象です。

**採点のしかた**

- `in` と `paraphrase`: 正解の語句（`must_include`）がすべて答えに含まれていれば正解です。
- `out`: 「記載がありません」「わかりません」などの語を含めば正解で、定義を作って答えた場合は `asserted`（断定）です。

**見るところ**: 条件ごとの 1 行の要約（`in_rate`、`paraphrase_rate`、`out_rate`）。評価は 1 条件あたり 1〜2 分かかります。

In [ ]:
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM
pd.set_option("display.max_colwidth", None)

DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
tok = AutoTokenizer.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"])
if tok.pad_token is None: tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"], dtype=torch.float32).to(DEVICE)
model.config.use_cache = True
print("loaded:", CONFIG["base_model"], "| device:", DEVICE)

def generate(m, messages):
    prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    x = tok(prompt, return_tensors="pt").to(m.device)
    with torch.no_grad():
        y = m.generate(**x, max_new_tokens=CONFIG["max_new_tokens"], pad_token_id=tok.pad_token_id, **CONFIG["gen"])
    return tok.decode(y[0][x["input_ids"].shape[1]:], skip_special_tokens=True).strip()

# ---------- 用語集検索（比較用。学習はしない） ----------
def names_of(t):
    '''見出し語を括弧・中黒で分けたもの（2 文字以上）と別名（3 文字以上。r や PR などは誤ヒットが多いので使わない）'''
    heads = [t["term"]] + [p.strip() for p in re.split(r"[（）()・/]", t["term"])]
    return {n for n in heads if len(n) >= 2} | {a for a in t.get("aliases", []) if len(a.strip()) >= 3}

def search(q, k=3):
    ql = q.lower()
    hits = [t for t in TERMS if any(n.lower() in ql for n in names_of(t))]
    hits.sort(key=lambda t: -max(len(n) for n in names_of(t) if n.lower() in ql))  # 長い名前で一致したものを優先
    return hits[:k]

SEARCH_SYSTEM = ("あなたは講座の用語集の回答係です。与えられた用語集の項目だけを根拠に答えてください。"
                 "用語集に該当する項目が無いときは「用語集に記載がありません」と答えてください。")

def search_messages(q):
    hits = search(q)
    ctx = "\n".join(f"- {t['term']}: {t['definition']}" for t in hits) or "（該当なし）"
    return [{"role": "system", "content": SEARCH_SYSTEM},
            {"role": "user", "content": f"用語集:\n{ctx}\n\n質問: {q}"}], [t["id"] for t in hits]

# ---------- 採点 ----------
def grade(t, raw):
    refused = any(w in raw for w in REFUSAL)
    if t["type"] == "out":
        return {"ok": refused, "verdict": "refused" if refused else "asserted"}
    ok = all(w.lower() in raw.lower() for w in t["must_include"])
    return {"ok": ok, "verdict": "correct" if ok else ("refused" if refused else "wrong")}

def mark(ok: bool) -> str:
    return "○" if ok else "×"

def evaluate(m, label, use_search=False):
    rows, t0 = [], time.time()
    for t in TESTS:
        hit_ids = None
        if use_search:
            msgs, hit_ids = search_messages(t["q"])
        else:
            msgs = [{"role": "user", "content": t["q"]}]  # 学習データと同じ形（system なし、user だけ）
        raw = generate(m, msgs)
        rows.append({"label": label, "id": t["id"], "type": t["type"], "q": t["q"], "raw": raw,
                     "search_hits": hit_ids, **grade(t, raw)})
    summary = {"label": label, "seconds": round(time.time() - t0, 1)}
    for ty in ["in", "paraphrase", "out"]:
        sub = [r for r in rows if r["type"] == ty]
        summary[f"{ty}_rate"] = round(sum(r["ok"] for r in sub) / len(sub), 3)
    summary["all_rate"] = round(sum(r["ok"] for r in rows) / len(rows), 3)
    print(json.dumps(summary, ensure_ascii=False))
    return summary, rows

In [ ]:
before_summary, before_rows = evaluate(model, "before")
search_summary, search_rows = evaluate(model, "search", use_search=True)
display(pd.DataFrame([{"区分": b["type"], "質問": b["q"], "学習前の出力": b["raw"], "学習前": mark(b["ok"]),
                       "検索した項目": s["search_hits"], "用語集検索の出力": s["raw"], "用語集検索": mark(s["ok"])}
                      for b, s in zip(before_rows, search_rows)]))

## 4. LoRA 学習（T4 で数分）

**やること**: 116 件の学習データで、演習と同じ LoRA 設定のまま 5 エポック（75 ステップ）学習します。学習データは既に `messages` 形式なので、`to_chat` での変換は要りません。

**見るところ**

- `trainable%`: 演習と同じく、学習するのは全体の 1% 未満です。
- `Training Loss` の表（5 ステップごと）: 3〜4 前後から始まり、0.4 前後まで下がります。演習より最終 loss が高いのは、定義文が長く、データの種類も多いためです。
- `学習時間: ... 秒 | ステップ数: 75`

In [ ]:
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

train_ds = Dataset.from_list(train_rows)   # 各行が messages 形式なのでそのまま渡せる
model.config.use_cache = False
lora = LoraConfig(r=CONFIG["lora_r"], lora_alpha=CONFIG["lora_alpha"], lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
                  target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
sft = SFTConfig(output_dir=str(WORK / "ckpt"), num_train_epochs=CONFIG["num_train_epochs"], per_device_train_batch_size=4,
                gradient_accumulation_steps=2, learning_rate=CONFIG["lr"], lr_scheduler_type="cosine",
                warmup_steps=CONFIG["warmup_steps"], logging_steps=5, save_strategy="no",
                max_length=CONFIG["max_seq_len"], packing=False, report_to="none", seed=CONFIG["seed"])
trainer = SFTTrainer(model=model, args=sft, train_dataset=train_ds, peft_config=lora, processing_class=tok)
trainer.model.print_trainable_parameters()
trainable, total = trainer.model.get_nb_trainable_parameters()
t0 = time.time()
trainer.train()
train_seconds = round(time.time() - t0, 1)
losses = [{"step": h["step"], "loss": round(h["loss"], 4)} for h in trainer.state.log_history if "loss" in h]
print(f"学習時間: {train_seconds:.0f} 秒 | ステップ数: {trainer.state.global_step} | 最終 loss: {losses[-1]['loss'] if losses else None}")
ADAPTER_DIR = WORK / "adapter"
trainer.model.save_pretrained(ADAPTER_DIR)
if USE_DRIVE:
    import shutil; shutil.copytree(ADAPTER_DIR, Path(CONFIG["drive_dir"]) / "adapter", dirs_exist_ok=True)
tuned = trainer.model
tuned.config.use_cache = True
_ = tuned.eval()

## 5. 学習後の評価と 3 条件の比較

**やること**: 学習後のモデルで、学習前と同じ 21 問・同じ生成設定で評価し、学習前・学習後・用語集検索の 3 条件を並べます。

**見るところ**

- 比較表: 区分ごとの正解率が条件によってどう違うか。
- `退行`: 学習前は正解だったのに、学習後に間違えた質問です。
- 質問ごとの表: 学習後の答えは定義文に似た形になります。正解の語句を含んでいても、作り話が混ざっていないかを読んでください。`out` の質問に定義を作って答えていないかも確かめます。

In [ ]:
after_summary, after_rows = evaluate(tuned, "after")
display(pd.DataFrame([before_summary, after_summary, search_summary]).set_index("label")[["in_rate", "paraphrase_rate", "out_rate", "all_rate", "seconds"]])
regressed = [b["id"] for b, a in zip(before_rows, after_rows) if b["ok"] and not a["ok"]]
print("退行:", regressed)
display(pd.DataFrame([{"区分": b["type"], "質問": b["q"],
                       "学習前": f"{mark(b['ok'])} {b['verdict']}", "学習後の出力": a["raw"], "学習後": f"{mark(a['ok'])} {a['verdict']}",
                       "用語集検索": f"{mark(s['ok'])} {s['verdict']}"}
                      for b, a, s in zip(before_rows, after_rows, search_rows)]))

## 6. 結果の保存と提出

**やること**: 設定、ライブラリの版、学習ログ、3 条件の要約、退行した質問を `result.json` に、全出力と採点を `eval_rows.json` にまとめます。

**提出するもの**: `/content/glossary_lab` の `result.json` と `eval_rows.json` をダウンロードし、`lectures/02/advanced/` に置きます。比較表と考察は `lectures/02/answer.md` に書きます。

In [ ]:
(WORK / "eval_rows.json").write_text(
    json.dumps({"before": before_rows, "after": after_rows, "search": search_rows}, ensure_ascii=False, indent=2), encoding="utf-8")
result = {
    "config": {k: v for k, v in CONFIG.items() if k not in ("work_dir", "drive_dir")},
    "versions": {m.__name__: m.__version__ for m in [transformers, peft, trl, torch]},
    "method": "LoRA（量子化なし）",
    "train": {"n": len(train_rows), "steps": trainer.state.global_step, "seconds": train_seconds,
              "trainable_params": trainable, "total_params": total,
              "trainable_pct": round(100 * trainable / total, 3), "losses": losses},
    "summaries": [before_summary, after_summary, search_summary],
    "regressed": regressed,
    "executed_training_myself": True,
}
(WORK / "result.json").write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps({k: v for k, v in result.items() if k != "train"}, ensure_ascii=False, indent=2))
if USE_DRIVE:
    import shutil
    for name in ["result.json", "eval_rows.json"]:
        shutil.copy(WORK / name, Path(CONFIG["drive_dir"]) / name)
    print("Drive に保存しました:", CONFIG["drive_dir"])